<a href="https://colab.research.google.com/github/snehag77/Recipe-Generator/blob/main/01_data_and_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q keras-hub datasets transformers

import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # must come before importing keras

import json, ast, random, re
import numpy as np, pandas as pd
import tensorflow as tf, keras, keras_hub
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/P45_recipe_gen"
for d in ["data", "outputs", "checkpoints", "logs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

random.seed(42); np.random.seed(42); tf.random.set_seed(42)
print("GPU:", tf.config.list_physical_devices("GPU"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: []


In [ ]:
# Only if you want to use the Kaggle API: upload kaggle.json first
# from google.colab import files; files.upload()
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d paultimothymooney/recipenlg -p {BASE}/data --unzip

In [ ]:
CSV = f"{BASE}/data/RecipeNLG_dataset.csv"
df = pd.read_csv(CSV, nrows=150_000, usecols=["title", "ingredients", "directions", "NER"])
print(df.shape)
df.head(2)

(150000, 4)


,title,ingredients,directions,NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""beef"", ""chicken breasts"", ""cream of mushroom..."


In [ ]:
def parse_list(x):
    try:
        v = ast.literal_eval(x)
        return v if isinstance(v, list) else None
    except Exception:
        return None

for col in ["ingredients", "directions", "NER"]:
    df[col] = df[col].apply(parse_list)

df = df.dropna().reset_index(drop=True)

def clean(s):
    return re.sub(r"\s+", " ", str(s)).strip()

df["title"] = df["title"].apply(clean)
df["NER"] = df["NER"].apply(lambda L: [clean(i).lower() for i in L if clean(i)])
df["ingredients"] = df["ingredients"].apply(lambda L: [clean(i) for i in L if clean(i)])
df["directions"] = df["directions"].apply(lambda L: [clean(i) for i in L if clean(i)])

# basic sanity filters
df = df[(df["NER"].str.len().between(2, 12)) &
        (df["ingredients"].str.len().between(2, 20)) &
        (df["directions"].str.len().between(1, 15)) &
        (df["title"].str.len().between(3, 80))].reset_index(drop=True)
print(len(df), "recipes after cleaning")

142289 recipes after cleaning


In [ ]:
def format_example(row, shuffle=True):
    """Full training string: ingredients -> title -> quantities -> steps."""
    ings = list(row["NER"])
    if shuffle:
        random.shuffle(ings)            # augmentation: model shouldn't depend on order
    steps = " ".join(f"{i+1}. {s}" for i, s in enumerate(row["directions"]))
    return (
        f"<INGREDIENTS> {', '.join(ings)} <END_ING>\n"
        f"<TITLE> {row['title']} <END_TITLE>\n"
        f"<QUANTITIES> {'; '.join(row['ingredients'])} <END_QTY>\n"
        f"<STEPS> {steps} <END_STEPS>"
    )

def build_prompt(ingredient_list):
    """Inference prompt: stops right after <TITLE> so the model writes the rest."""
    ings = [i.strip().lower() for i in ingredient_list if i.strip()]
    return f"<INGREDIENTS> {', '.join(ings)} <END_ING>\n<TITLE>"

def parse_output(text):
    """Split a generated string into fields (used later for checks and the app)."""
    def grab(start, end):
        m = re.search(rf"{re.escape(start)}(.*?){re.escape(end)}", text, re.S)
        return m.group(1).strip() if m else None
    return {
        "title": grab("<TITLE>", "<END_TITLE>"),
        "quantities": grab("<QUANTITIES>", "<END_QTY>"),
        "steps": grab("<STEPS>", "<END_STEPS>"),
    }

# look at one example
print(format_example(df.iloc[0]))
print("\n--- inference prompt ---")
print(build_prompt(["chicken", "rice", "garlic", "soy sauce"]))

<INGREDIENTS> nuts, milk, vanilla, butter, brown sugar, bite size shredded rice biscuits <END_ING>
<TITLE> No-Bake Nut Cookies <END_TITLE>
<QUANTITIES> 1 c. firmly packed brown sugar; 1/2 c. evaporated milk; 1/2 tsp. vanilla; 1/2 c. broken nuts (pecans); 2 Tbsp. butter or margarine; 3 1/2 c. bite size shredded rice biscuits <END_QTY>
<STEPS> 1. In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine. 2. Stir over medium heat until mixture bubbles all over top. 3. Boil and stir 5 minutes more. Take off heat. 4. Stir in vanilla and cereal; mix well. 5. Using 2 teaspoons, drop and shape into 30 clusters on wax paper. 6. Let stand until firm, about 30 minutes. <END_STEPS>

--- inference prompt ---
<INGREDIENTS> chicken, rice, garlic, soy sauce <END_ING>
<TITLE>


In [ ]:
from transformers import GPT2TokenizerFast
tok = GPT2TokenizerFast.from_pretrained("gpt2")
MAX_LEN = 256

df["text"] = df.apply(format_example, axis=1)
lens = [len(x) for x in tok(df["text"].tolist(), add_special_tokens=False)["input_ids"]]
df["n_tokens"] = lens

print(df["n_tokens"].describe())
df = df[df["n_tokens"] <= MAX_LEN - 2].reset_index(drop=True)   # leave room for start/end tokens
print(len(df), "recipes fit in", MAX_LEN, "tokens")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

count    142289.000000
mean        186.030305
std          48.277722
min          63.000000
25%         151.000000
50%         181.000000
75%         215.000000
max         496.000000
Name: n_tokens, dtype: float64
129645 recipes fit in 256 tokens


In [ ]:
N = min(30_000, len(df))                       # subset size for free Colab
df = df.sample(n=N, random_state=42).reset_index(drop=True)

n_train, n_val = int(0.90 * N), int(0.05 * N)
splits = {
    "train": df.iloc[:n_train],
    "val":   df.iloc[n_train:n_train + n_val],
    "test":  df.iloc[n_train + n_val:],
}

for name, part in splits.items():
    path = f"{BASE}/data/{name}.jsonl"
    with open(path, "w") as f:
        for _, r in part.iterrows():
            f.write(json.dumps({
                "title": r["title"], "NER": r["NER"],
                "ingredients": r["ingredients"], "directions": r["directions"],
                "text": r["text"],
            }) + "\n")
    print(name, len(part), "->", path)

train 27000 -> /content/drive/MyDrive/P45_recipe_gen/data/train.jsonl
val 1500 -> /content/drive/MyDrive/P45_recipe_gen/data/val.jsonl
test 1500 -> /content/drive/MyDrive/P45_recipe_gen/data/test.jsonl


In [ ]:
TEST_SETS = [
    ["chicken", "rice", "garlic", "soy sauce"],                  # simple meat dish
    ["paneer", "tomato", "onion", "cream", "garam masala"],      # Indian-style
    ["flour", "butter", "sugar", "eggs", "vanilla"],             # baking
    ["potato", "spinach", "chickpeas", "cumin"],                 # vegetarian/vegan
    ["banana", "peanut butter", "oats", "honey"],                # sweet, with allergen (peanut)
]
with open(f"{BASE}/outputs/test_sets.json", "w") as f:
    json.dump(TEST_SETS, f, indent=2)

In [ ]:
preprocessor = keras_hub.models.GPT2CausalLMPreprocessor.from_preset(
    "gpt2_base_en", sequence_length=MAX_LEN)
base_model = keras_hub.models.GPT2CausalLM.from_preset(
    "gpt2_base_en", preprocessor=preprocessor)

# Fixed sampler, reused later for the fine-tuned model so the comparison is fair
SAMPLER = keras_hub.samplers.TopKSampler(k=40, temperature=0.8, seed=42)
base_model.compile(sampler=SAMPLER)

def generate_recipe(model, ingredients, max_length=MAX_LEN):
    prompt = build_prompt(ingredients)
    out = model.generate(prompt, max_length=max_length)
    return out if isinstance(out, str) else out[0]

base_outputs = []
for ings in TEST_SETS:
    text = generate_recipe(base_model, ings)
    base_outputs.append({"ingredients": ings, "output": text})
    print("=" * 70)
    print("INPUT:", ings)
    print(text)

INPUT: ['chicken', 'rice', 'garlic', 'soy sauce']
<INGREDIENTS> chicken, rice, garlic, soy sauce <END_ING>
<TITLE>Mushrooms <END_TITLE>Sugar <END_TITLE>Tofu <END_TITLE>
<END_TITLE>Kimchi (sugar) <END_TITLE>
<END_TITLE>Pasta (pasta) <END_TITLE>
<END_TITLE>Pasta with sugar <END_TITLE>
<END_TITLE>Pasta with soy sauce <END_TITLE>
<END_TITLE>Pasta with soy sauce with no sugar <END_TITLE>
<END_TITLE>Pasta with soy sauce with soy sauce with no sugar <END_TITLE>
<END_TITLE>Pasta with soy sauce with soy sauce with a no sugar <END_TITLE>
<END_TITLE>Pasta with soy sauce with soy sauce with soy sauce with soy sauce with soy sauce with soy sauce with soy sauce <END_TITLE>Pasta with soy sauce with soy sauce with soy sauce with soy sauce with soy sauce <END_TITLE>Pasta
INPUT: ['paneer', 'tomato', 'onion', 'cream', 'garam masala']
<INGREDIENTS> paneer, tomato, onion, cream, garam masala <END_ING>
<TITLE> The Essential Cooking Cookbook <BRAND_NUMBER>
<BRAND_NAME> Cooking Cookbooks <BRAND_REDUCED_TITLE>

In [ ]:
with open(f"{BASE}/outputs/base_outputs.json", "w") as f:
    json.dump(base_outputs, f, indent=2)

REQUIRED = ["<TITLE>", "<END_TITLE>", "<QUANTITIES>", "<END_QTY>", "<STEPS>", "<END_STEPS>"]
def format_adherence(text):
    return sum(tag in text for tag in REQUIRED) / len(REQUIRED)

def ingredient_coverage(ings, text):
    t = text.lower()
    return sum(i.lower() in t for i in ings) / len(ings)

for o in base_outputs:
    print(o["ingredients"],
          "| format:", f"{format_adherence(o['output']):.0%}",
          "| coverage:", f"{ingredient_coverage(o['ingredients'], o['output']):.0%}")

['chicken', 'rice', 'garlic', 'soy sauce'] | format: 33% | coverage: 100%
['paneer', 'tomato', 'onion', 'cream', 'garam masala'] | format: 17% | coverage: 100%
['flour', 'butter', 'sugar', 'eggs', 'vanilla'] | format: 17% | coverage: 100%
['potato', 'spinach', 'chickpeas', 'cumin'] | format: 17% | coverage: 100%
['banana', 'peanut butter', 'oats', 'honey'] | format: 17% | coverage: 100%
